# ISCO Classification - OpenAI Batch API Interaction

This notebook manages ISCO classification batch submissions to OpenAI.

Workflow:
1. Submit JSONL files from `openai_inputs/` to OpenAI Batch API
2. Check batch status
3. Download completed results to `openai_responses/`

In [7]:
import sys
import yaml
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent.parent.parent
sys.path.insert(0, str(project_root / "src"))

from bonn_thesis.openai_processing.isco_batch_manager import (
    check_status,
    download_batch_results,
    submit_batch,
)
from bonn_thesis.config import OCCUPATION_DATA_BLD, SRC

## 1. Submit a Batch

Submit a single JSONL file to OpenAI. You can run this cell multiple times for different batch files.

In [8]:
# Load experiment configuration
experiment_config_path = (
    SRC / "openai_processing" / "configs" / "occupation_classification" / "occupation_classification_001.yaml"
)
with open(experiment_config_path) as f:
    experiment_config = yaml.safe_load(f)

# Paths
input_dir = OCCUPATION_DATA_BLD / "openai_inputs"
metadata_csv_path = OCCUPATION_DATA_BLD / "isco_classification_metadata.csv"

# Select which batch to submit (change the number as needed)
batch_number = "test_data_base_model"  # Change this to submit different batches
jsonl_path = input_dir / f"isco_classification_{batch_number}.jsonl"

print(f"Submitting batch: {jsonl_path.name}")

# Submit batch
result = submit_batch(
    jsonl_path=str(jsonl_path),
    experiment_config=experiment_config,
    metadata_csv_path=str(metadata_csv_path),
)

# Save the batch ID for later use
batch_id = result["batch_id"]
print("\n✓ Batch submitted!")
print(f"Batch ID: {batch_id}")
print(f"Batch Name: {result['batch_name']}")

Submitting batch: isco_classification_test_data_base_model.jsonl
SUBMITTING ISCO CLASSIFICATION BATCH: isco_classification_test_data_base_model
Uploading file: /Users/willbackes/Documents/code/bonn_thesis/bld/data/occupation_data/openai_inputs/isco_classification_test_data_base_model.jsonl
File uploaded successfully. File ID: file-XDwcipg9z7C7GD9xRLGdjP
Creating batch: isco_classification_test_data_base_model
Batch created successfully. Batch ID: batch_697f797ddd108190949babf06a098227
Status: validating
SUBMISSION COMPLETE
Batch Name: isco_classification_test_data_base_model
Batch ID: batch_697f797ddd108190949babf06a098227
File ID: file-XDwcipg9z7C7GD9xRLGdjP

✓ Batch submitted!
Batch ID: batch_697f797ddd108190949babf06a098227
Batch Name: isco_classification_test_data_base_model


## 2. Submit Multiple Batches

Submit multiple batches at once. Adjust the range as needed.

In [ ]:
# Load experiment configuration
experiment_config_path = (
    SRC / "openai_processing" / "configs" / "occupation_classification" / "occupation_classification_001.yaml"
)
with open(experiment_config_path) as f:
    experiment_config = yaml.safe_load(f)

# Paths
input_dir = OCCUPATION_DATA_BLD / "openai_inputs"
metadata_csv_path = OCCUPATION_DATA_BLD / "isco_classification_metadata.csv"

# Get all JSONL files
jsonl_files = sorted(input_dir.glob("isco_classification_*.jsonl"))
print(f"Found {len(jsonl_files)} JSONL files")

# Submit batches (adjust range as needed)
batch_ids = []
start_idx = 0  # Start from first file
end_idx = 10   # Submit first 10 files (adjust as needed)

for jsonl_path in jsonl_files[start_idx:end_idx]:
    # Skip empty files
    if jsonl_path.stat().st_size == 0:
        print(f"Skipping empty file: {jsonl_path.name}")
        continue
    
    print(f"\nSubmitting: {jsonl_path.name}")
    
    try:
        result = submit_batch(
            jsonl_path=str(jsonl_path),
            experiment_config=experiment_config,
            metadata_csv_path=str(metadata_csv_path),
        )
        
        batch_ids.append({
            "batch_id": result["batch_id"],
            "batch_name": result["batch_name"],
            "file": jsonl_path.name,
        })
        
        print(f"  ✓ Submitted: {result['batch_id']}")
    except Exception as e:
        print(f"  ✗ Error: {e}")

print(f"\n✓ Submitted {len(batch_ids)} batches successfully")
print("\nBatch IDs:")
for info in batch_ids:
    print(f"  {info['file']}: {info['batch_id']}")

## 3. Check Batch Status

Check the status of a submitted batch.

In [10]:
# Replace with your batch ID
batch_id = "batch_697f797ddd108190949babf06a098227"  # Paste your batch ID here

status_info = check_status(batch_id)

# Print status
current_status = status_info["status"]
print(f"\nCurrent status: {current_status}")

if current_status == "completed":
    print("\n✓ Batch is complete! Ready to download results.")
elif current_status in ["validating", "in_progress", "finalizing"]:
    print("\n⏳ Batch is still processing. Check back later.")
elif current_status == "failed":
    print("\n❌ Batch failed. Check error logs.")
elif current_status == "expired":
    print("\n⚠️ Batch expired without completing.")

BATCH STATUS: COMPLETED
Batch ID: batch_697f797ddd108190949babf06a098227
Batch Name: isco_classification_test_data_base_model

Progress:
  Total requests: 300
  Completed: 300
  Failed: 0
  Progress: 100.0%

Timestamps:
  Created: 2026-02-01 17:04:13
  Started: 2026-02-01 17:05:15
  Completed: 2026-02-01 17:07:41

Current status: completed

✓ Batch is complete! Ready to download results.


## 4. Check Multiple Batch Statuses

Check status of all submitted batches from metadata CSV.

In [ ]:
import pandas as pd

# Load metadata to get all batch IDs
metadata_csv_path = OCCUPATION_DATA_BLD / "isco_classification_metadata.csv"
metadata = pd.read_csv(metadata_csv_path)

# Filter for batches that have been submitted (have batch_id)
submitted_batches = metadata[metadata["batch_id"].notna()].copy()

print(f"Checking status for {len(submitted_batches)} batches...\n")

# Check each batch
statuses = []
for idx, row in submitted_batches.iterrows():
    batch_id = row["batch_id"]
    batch_name = row["batch_name"]
    
    try:
        status_info = check_status(batch_id)
        status = status_info["status"]
        statuses.append({"batch_name": batch_name, "batch_id": batch_id, "status": status})
    except Exception as e:
        statuses.append({"batch_name": batch_name, "batch_id": batch_id, "status": f"Error: {e}"})

# Create summary DataFrame
status_df = pd.DataFrame(statuses)
print("\nBatch Status Summary:")
print(status_df.to_string(index=False))

# Count by status
print("\nStatus Counts:")
print(status_df["status"].value_counts())

## 5. Download Batch Results

Download completed batch results to `openai_responses/`.

In [11]:
# Replace with your batch ID
batch_id = "batch_697f797ddd108190949babf06a098227"  # Paste your batch ID here

# Download results
output_dir = OCCUPATION_DATA_BLD / "openai_responses"

try:
    result = download_batch_results(batch_id, str(output_dir))
    print("\n✓ Results downloaded successfully!")
    print(f"Saved to: {result['output_path']}")
    print(f"Batch name: {result['batch_name']}")
    print(f"Number of responses: {result['n_responses']}")
except ValueError as e:
    print(f"\n❌ Error: {e}")
    print("Make sure the batch is completed before downloading.")


Results saved to: /Users/willbackes/Documents/code/bonn_thesis/bld/data/occupation_data/openai_responses/isco_classification_test_data_base_model_results.jsonl
Total responses: 300

✓ Results downloaded successfully!
Saved to: /Users/willbackes/Documents/code/bonn_thesis/bld/data/occupation_data/openai_responses/isco_classification_test_data_base_model_results.jsonl
Batch name: isco_classification_test_data_base_model
Number of responses: 300


## 6. Download All Completed Batches

Download results for all completed batches.

In [ ]:
import pandas as pd

# Load metadata to get all batch IDs
metadata_csv_path = OCCUPATION_DATA_BLD / "isco_classification_metadata.csv"
metadata = pd.read_csv(metadata_csv_path)

# Filter for submitted batches
submitted_batches = metadata[metadata["batch_id"].notna()].copy()

print(f"Checking {len(submitted_batches)} batches for completed results...\n")

output_dir = OCCUPATION_DATA_BLD / "openai_responses"
downloaded = []
skipped = []

for idx, row in submitted_batches.iterrows():
    batch_id = row["batch_id"]
    batch_name = row["batch_name"]
    
    try:
        # Check status first
        status_info = check_status(batch_id)
        status = status_info["status"]
        
        if status == "completed":
            # Download results
            result = download_batch_results(batch_id, str(output_dir))
            downloaded.append({
                "batch_name": batch_name,
                "n_responses": result["n_responses"],
                "file": result["output_path"],
            })
            print(f"✓ Downloaded: {batch_name} ({result['n_responses']} responses)")
        else:
            skipped.append({"batch_name": batch_name, "status": status})
            print(f"⏸ Skipped: {batch_name} (status: {status})")
    
    except Exception as e:
        print(f"✗ Error downloading {batch_name}: {e}")
        skipped.append({"batch_name": batch_name, "status": f"Error: {e}"})

print(f"\n✓ Downloaded {len(downloaded)} batches")
print(f"⏸ Skipped {len(skipped)} batches")

if downloaded:
    print("\nDownloaded batches:")
    for info in downloaded:
        print(f"  {info['batch_name']}: {info['n_responses']} responses")